In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 81.4 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
!nvidia-smi

GPU Available: True
Fri Aug  7 00:32:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

In [ ]:
!ls "/content/drive/MyDrive/cv_project/dataset"
!cat "/content/drive/MyDrive/cv_project/dataset/data.yaml"

data.yaml  images  labels
path: /content/drive/MyDrive/cv_project/dataset
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: Hardhat
  1: NO-Hardhat
  2: Person


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/drive/MyDrive/cv_project/dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    pretrained=True,
    project="/content/drive/MyDrive/cv_project/outputs/training_results",
    name="yolov8n_baseline"
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/cv_project/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, flip

In [ ]:
!cp "/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/weights/best.pt" "/content/drive/MyDrive/cv_project/models/best.pt"

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/cv_project/models/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/cv_project/models/best.pt")

metrics = model.val(
    data="/content/drive/MyDrive/cv_project/dataset/data.yaml",
    split="test",
    project="/content/drive/MyDrive/cv_project/outputs/training_results",
    name="test_eval"
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.0±0.7 ms, read: 0.1±0.1 MB/s, size: 96.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/cv_project/dataset/labels/test... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 1.7s/it 34.5s
val: New cache created: /content/drive/MyDrive/cv_project/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3s/it 2.6s
                   all         20         85      0.598      0.473      0.476       0.26
               Hardhat         11         29      0.804       0.69      0.707      0.366
            NO-Hardhat          6         10      0.287      0.165  

In [ ]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

print("\nPer-class mAP@0.5:")
for i, name in model.names.items():
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")

Precision: 0.5984434678008862
Recall: 0.4733069923027385
mAP@0.5: 0.4763375489014075
mAP@0.5:0.95: 0.2601002754822743

Per-class mAP@0.5:
  Hardhat: 0.7069
  NO-Hardhat: 0.1003
  Person: 0.6218


In [ ]:
!ls "/content/drive/MyDrive/cv_project/outputs/training_results/yolov8n_baseline/"

args.yaml			 results.csv	     val_batch0_labels.jpg
BoxF1_curve.png			 results.png	     val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg    val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg    val_batch1_pred.jpg
BoxR_curve.png			 train_batch2.jpg    val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch360.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch361.jpg  weights
labels.jpg			 train_batch362.jpg


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/cv_project/models/best.pt")

results = model.predict(
    source="/content/drive/MyDrive/cv_project/dataset/images/test",
    conf=0.35,
    save=True,
    project="/content/drive/MyDrive/cv_project/outputs/predictions",
    name="test_predictions"
)


image 1/20 /content/drive/MyDrive/cv_project/dataset/images/test/-1969-_png_jpg.rf.c383b0a9a12f96b66d8089f4b4cb8b5c.jpg: 640x640 1 Hardhat, 7.2ms
image 2/20 /content/drive/MyDrive/cv_project/dataset/images/test/003184_jpg.rf.350907e4e78ed835b42f0aca59001d5a.jpg: 640x448 4 Hardhats, 3 NO-Hardhats, 4 Persons, 70.0ms
image 3/20 /content/drive/MyDrive/cv_project/dataset/images/test/2008_008337_jpg.rf.43e82a2e5b55a85a4106caacc8da8401.jpg: 480x640 1 Person, 73.4ms
image 4/20 /content/drive/MyDrive/cv_project/dataset/images/test/2008_008753_jpg.rf.8d06fcf951f7e512797e9a30ad3f8747.jpg: 480x640 2 Persons, 28.0ms
image 5/20 /content/drive/MyDrive/cv_project/dataset/images/test/2009_000379_jpg.rf.350a21e166ddfda05b3949f749566483.jpg: 480x640 1 Person, 15.9ms
image 6/20 /content/drive/MyDrive/cv_project/dataset/images/test/IMG_3093_mp4-22_jpg.rf.dc13f7b0b5e4bc0383a1af58a9df93b9.jpg: 640x384 1 NO-Hardhat, 2 Persons, 65.7ms
image 7/20 /content/drive/MyDrive/cv_project/dataset/images/test/autox4_mp4

In [ ]:
!zip -r /content/test_predictions.zip "/content/drive/MyDrive/cv_project/outputs/predictions/test_predictions"

from google.colab import files
files.download("/content/test_predictions.zip")

  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/ (stored 0%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/-1969-_png_jpg.rf.c383b0a9a12f96b66d8089f4b4cb8b5c.jpg (deflated 4%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/003184_jpg.rf.350907e4e78ed835b42f0aca59001d5a.jpg (deflated 3%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/2008_008337_jpg.rf.43e82a2e5b55a85a4106caacc8da8401.jpg (deflated 4%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/2008_008753_jpg.rf.8d06fcf951f7e512797e9a30ad3f8747.jpg (deflated 5%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/2009_000379_jpg.rf.350a21e166ddfda05b3949f749566483.jpg (deflated 5%)
  adding: content/drive/MyDrive/cv_project/outputs/predictions/test_predictions/IMG_3093_mp4-22_jpg.rf.dc13f7b0b5e4bc0383a1af58a9df93b9.jpg (deflated 7%)
  adding:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>